In [25]:
import os
import json
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
import numpy as np
import pandas as pd

def read_json(path):
    with open(path, 'r', encoding="utf-8") as f:
        data = json.load(f)
    return data

def write_json(data, path):
    if not os.path.exists(os.path.dirname(path)):
        
        os.makedirs(os.path.dirname(path))
    with open(path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [26]:
import numpy as np

def compute_forgetting_matrix(accuracy_matrix):
    """
    Computes forgetting for each task.
    
    Parameters:
    - accuracy_matrix: 2D numpy array of shape (num_tasks, num_tasks)
      Each row i represents accuracy on task i after training on tasks j=0..T-1
    
    Returns:
    - forgetting: list of forgetting values for each task
    - avg_forgetting: average forgetting across tasks (excluding the last task)
    """
    num_tasks = accuracy_matrix.shape[0]
    forgetting = []

    for i in range(num_tasks - 1):
        max_acc = np.max(accuracy_matrix[i, :i+1])  # Max accuracy before final training
        final_acc = accuracy_matrix[i, -1]          # Accuracy after training on all tasks
        forgetting.append(max_acc - final_acc)

    avg_forgetting = np.mean(forgetting)
    return forgetting, avg_forgetting


In [18]:
!pip install simplemma




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 MB 4.4 MB/s eta 0:00:0000:0100:01m


In [27]:

import simplemma
from simplemma import text_lemmatizer
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def canonicalize(s: str) -> str:
    if s == "":
        return s
    s = s.strip().lower()
    s = lemmatizer.lemmatize(s, pos=wordnet.VERB)
    if s.endswith("ies"):           # e.g., "subsidiaries" -> "subsidiary"
        # print(s)
        return s[:-3] + "y"
    if s.endswith("s") and not s.endswith("ss"):
        # print(s)
        return s[:-1]               # crude plural -> singular
    return s


[nltk_data] Downloading package wordnet to /Users/sefika/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/sefika/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [28]:
# matrices preparation on T5
def compute_forgetting_matrix_t5(input_folder, test_folder):
    results = []
    for i in range(1, 9):
       input_file = f"/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas/{input_folder}/test_pred_{i}.json"
       start=0 
       predictions = read_json(input_file)
       predictions = [canonicalize(item['predict']) for item in predictions]
       
       for id in range(1, i+1):
            test_file = f"/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/fewrel/test/{test_folder}/task{id}/test.json"
            test_data = read_json(test_file)
            y_true = [canonicalize(item['relation']) for item in test_data]
            y_task = predictions[start:start+len(y_true)]
            acc = accuracy_score(y_true, y_task)
            print(f"Task {i} - Task {id} Accuracy: {acc:.4f}")
            print(f"Task {i} - Task {id} Size: {len(y_task)}")
            print(f"Task {i} - Task {id} True Size: {len(y_true)}")

            row = {'base_task':i , 'task':id, 'accuracy':acc}
        
            start += len(y_true)
            results.append(row)
    return results

for i in range(1, 6):
    results = compute_forgetting_matrix_t5(
        input_folder=f"model_{i}",
        test_folder=f"run{i}"
    )
    results
    write_json(results, f"/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas/canonical/model_{i}_forgetting_matrix.json")

Task 1 - Task 1 Accuracy: 0.9821
Task 1 - Task 1 Size: 1400
Task 1 - Task 1 True Size: 1400
Task 2 - Task 1 Accuracy: 0.9800
Task 2 - Task 1 Size: 1400
Task 2 - Task 1 True Size: 1400
Task 2 - Task 2 Accuracy: 0.8543
Task 2 - Task 2 Size: 1400
Task 2 - Task 2 True Size: 1400
Task 3 - Task 1 Accuracy: 0.9700
Task 3 - Task 1 Size: 1400
Task 3 - Task 1 True Size: 1400
Task 3 - Task 2 Accuracy: 0.8536
Task 3 - Task 2 Size: 1400
Task 3 - Task 2 True Size: 1400
Task 3 - Task 3 Accuracy: 0.8557
Task 3 - Task 3 Size: 1400
Task 3 - Task 3 True Size: 1400
Task 4 - Task 1 Accuracy: 0.9657
Task 4 - Task 1 Size: 1400
Task 4 - Task 1 True Size: 1400
Task 4 - Task 2 Accuracy: 0.8343
Task 4 - Task 2 Size: 1400
Task 4 - Task 2 True Size: 1400
Task 4 - Task 3 Accuracy: 0.8393
Task 4 - Task 3 Size: 1400
Task 4 - Task 3 True Size: 1400
Task 4 - Task 4 Accuracy: 0.8814
Task 4 - Task 4 Size: 1400
Task 4 - Task 4 True Size: 1400
Task 5 - Task 1 Accuracy: 0.9636
Task 5 - Task 1 Size: 1400
Task 5 - Task 1 True

In [5]:
# matrices preparation on T5
def compute_forgetting_matrix_t5(input_folder, test_folder):
    results = []
    for i in range(1, 9):
       input_file = f"/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/baseline/{input_folder}/test_pred_{i}.json"
       start=0 
       predictions = read_json(input_file)
       predictions = [item['predict'] for item in predictions]
       
       for id in range(1, i+1):
        test_file = f"/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/fewrel/test/{test_folder}/task{id}/test.json"
        test_data = read_json(test_file)
        y_true = [item['relation'] for item in test_data]
        y_task = predictions[start:start+len(y_true)]
        acc = accuracy_score(y_true, y_task)
        print(f"Task {i} - Task {id} Accuracy: {acc:.4f}")
        print(f"Task {i} - Task {id} Size: {len(y_task)}")
        print(f"Task {i} - Task {id} True Size: {len(y_true)}")

        row = {'base_task':i , 'task':id, 'accuracy':acc}
        
        start += len(y_true)
        results.append(row)
    return results
model="model_1"
for i in range(1, 6):
    results = compute_forgetting_matrix_t5(
        input_folder=f"model_{i}",
        test_folder=f"run{i}"
    )
    results
    write_json(results, f"/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/baseline/model_{i}_forgetting_matrix.json")

Task 1 - Task 1 Accuracy: 0.9864
Task 1 - Task 1 Size: 1400
Task 1 - Task 1 True Size: 1400
Task 2 - Task 1 Accuracy: 0.9686
Task 2 - Task 1 Size: 1400
Task 2 - Task 1 True Size: 1400
Task 2 - Task 2 Accuracy: 0.8621
Task 2 - Task 2 Size: 1400
Task 2 - Task 2 True Size: 1400
Task 3 - Task 1 Accuracy: 0.9650
Task 3 - Task 1 Size: 1400
Task 3 - Task 1 True Size: 1400
Task 3 - Task 2 Accuracy: 0.8386
Task 3 - Task 2 Size: 1400
Task 3 - Task 2 True Size: 1400
Task 3 - Task 3 Accuracy: 0.8457
Task 3 - Task 3 Size: 1400
Task 3 - Task 3 True Size: 1400
Task 4 - Task 1 Accuracy: 0.9529
Task 4 - Task 1 Size: 1400
Task 4 - Task 1 True Size: 1400
Task 4 - Task 2 Accuracy: 0.8171
Task 4 - Task 2 Size: 1400
Task 4 - Task 2 True Size: 1400
Task 4 - Task 3 Accuracy: 0.8264
Task 4 - Task 3 Size: 1400
Task 4 - Task 3 True Size: 1400
Task 4 - Task 4 Accuracy: 0.8857
Task 4 - Task 4 Size: 1400
Task 4 - Task 4 True Size: 1400
Task 5 - Task 1 Accuracy: 0.9571
Task 5 - Task 1 Size: 1400
Task 5 - Task 1 True

In [ ]:
# matrices preparation on Llama-2